In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from torch_geometric.nn import GCNConv
from transformers import AutoTokenizer
from torch_geometric.utils import add_self_loops
import json
import os
import evaluate  # Requires: pip install evaluate

# ==================================================================================
# CONFIGURATION
# ==================================================================================

# Path to the weights file (e.g., 'eeg-text-max-bleu.pt')
MODEL_WEIGHTS_PATH = "eeg-text-polished.pt" 

# Paths
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

# Dimensions
NUM_COLORS = 9       
NUM_OBJECTS = 6      
COLOR_NAMES = ["Black", "Blue", "Brown", "Green", "Grey", "Orange", "Red", "White", "Yellow"]
OBJECT_NAMES = ["Animal", "Building", "Food", "Nature", "Person", "Vehicle"]
ENC_HIDDEN = 256
DEC_HIDDEN = 256
DEC_LAYERS = 2
EMB_DIM = 256

# CPU Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Load Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    print("Local tokenizer not found, downloading bert-base-uncased...")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# ==================================================================================
# MODEL CLASSES
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, 
                          dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size, _, num_timesteps = eeg.shape
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        x = F.relu(self.gcn2(self.dropout(F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))), batch_edge_index, batch_edge_attr))
        
        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        return encoder_outputs.permute(1, 0, 2), encoder_hidden

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        self.color_processor = nn.Sequential(nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, color_feature_dim))
        self.object_processor = nn.Sequential(nn.Linear(num_objects, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, object_feature_dim))
        self.output_dim = color_feature_dim + object_feature_dim

    def forward(self, metadata):
        return torch.cat([self.color_processor(metadata[:, :NUM_COLORS].float()), 
                          self.object_processor(metadata[:, NUM_COLORS:].float())], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = nn.Linear(enc_hidden * 2, dec_hidden) 
        self.rnn = nn.GRU(emb_dim + enc_hidden * 2 + meta_features_dim + enc_hidden * 2, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_hidden * 2, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last = torch.cat((hidden[-1][0], hidden[-1][1]), dim=1)
        return torch.tanh(self.bridge(last)).unsqueeze(0).repeat(self.num_layers, 1, 1)

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        embedded = self.dropout(self.embedding(token.unsqueeze(0)))
        attn_energies = self.attention(encoder_outputs)
        scores = torch.bmm(decoder_hidden[-1].unsqueeze(0).permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        context = torch.bmm(F.softmax(scores, dim=2), encoder_outputs.permute(1, 0, 2)).permute(1, 0, 2)
        
        rnn_input = torch.cat((embedded, context, meta_features.unsqueeze(0), global_eeg_context.unsqueeze(0)), dim=2)
        output, hidden = self.rnn(rnn_input, decoder_hidden)
        return self.fc_out(output.squeeze(0)), hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_feature_dim=32, object_feature_dim=32, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        self.meta_encoder = MetadataEncoder(num_colors, num_objects, color_feature_dim, object_feature_dim)
        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden, self.meta_encoder.output_dim, dec_layers, pad_id, dropout)
        self.meta_head = nn.Sequential(nn.Linear(enc_hidden*2, 256), nn.ReLU(), nn.LayerNorm(256), nn.Dropout(0.3), nn.Linear(256, num_colors + num_objects))
        self.num_colors = num_colors

# ==================================================================================
# HELPERS
# ==================================================================================

def create_static_graph(num_channels=62):
    edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
    edge_attr = torch.ones(edge_index.shape[1], dtype=torch.float)
    return edge_index, edge_attr

def beam_search_decoder(model, eeg_signal, meta_signal, edge_index, edge_attr, beam_width=3, max_len=30):
    model.eval()
    with torch.no_grad():
        eeg, meta = eeg_signal.unsqueeze(0).to(device), meta_signal.unsqueeze(0).to(device)
        enc_out, enc_hid = model.encoder(eeg, edge_index, edge_attr)
        meta_feat = model.meta_encoder(meta)
        dec_hid = model.decoder.init_hidden(enc_hid)
        
        hidden_reshaped = enc_hid.view(model.encoder.rnn.num_layers, 2, 1, -1)
        global_ctx = torch.cat((hidden_reshaped[-1][0], hidden_reshaped[-1][1]), dim=1)
        
        meta_preds = model.meta_head(global_ctx)
        pred_color_ids = (torch.sigmoid(meta_preds[0, :model.num_colors]) > 0.5).nonzero(as_tuple=True)[0].tolist()
        pred_object_ids = (torch.sigmoid(meta_preds[0, model.num_colors:]) > 0.5).nonzero(as_tuple=True)[0].tolist()

        beams = [(0.0, SOS_ID, dec_hid, [])]
        
        for _ in range(max_len):
            candidates = []
            for score, inp, hid, seq in beams:
                if len(seq) > 0 and seq[-1] == EOS_ID:
                    candidates.append((score, inp, hid, seq)); continue
                
                # === FIX: Corrected variable name from 'input_id' to 'inp' ===
                token_tensor = torch.tensor([inp], device=device)
                
                pred, new_hid, _ = model.decoder(token_tensor, hid, enc_out, meta_feat, global_ctx)
                
                log_probs = F.log_softmax(pred, dim=-1).squeeze(0)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for k in range(beam_width):
                    candidates.append((score + topk_probs[k].item(), topk_ids[k].item(), new_hid, seq + [topk_ids[k].item()]))
            
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(seq[-1] == EOS_ID for _, _, _, seq in beams if len(seq) > 0): break

        best_seq = beams[0][3][:-1] if beams[0][3] and beams[0][3][-1] == EOS_ID else beams[0][3]
        return tokenizer.decode(best_seq, skip_special_tokens=True), pred_color_ids, pred_object_ids

# ==================================================================================
# MAIN EXECUTION
# ==================================================================================
if __name__ == "__main__":
    if not os.path.exists(MODEL_WEIGHTS_PATH):
        print(f"Error: Weights file '{MODEL_WEIGHTS_PATH}' not found.")
        exit()

    # 1. Load Metrics
    print("Loading evaluation metrics...")
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")

    # 2. Load Model
    print("Loading model on CPU...")
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        enc_hidden=ENC_HIDDEN,
        dec_hidden=DEC_HIDDEN,
        emb_dim=EMB_DIM,
        dec_layers=DEC_LAYERS
    ).to(device)

    try:
        state_dict = torch.load(MODEL_WEIGHTS_PATH, map_location=device)
        model.load_state_dict(state_dict)
        print("Model loaded successfully.")
    except Exception as e:
        print(f"CRITICAL: Model mismatch error.\n{e}")
        exit()

    # 3. Graph
    edge_index, edge_attr = create_static_graph(num_channels=62)
    edge_index, edge_attr = edge_index.to(device), edge_attr.to(device)

    # 4. Inference Loop
    print("Loading dataset...")
    with h5py.File(H5_FILE_PATH, 'r') as f:
        total_samples = f['eeg'].shape[0]
        NUM_SAMPLES = 1400  # Number of samples to evaluate
        
        predictions = []
        references = []

        print(f"\nRunning inference on {NUM_SAMPLES} random samples...")
        
        for n in range(NUM_SAMPLES):
            idx = np.random.randint(0, total_samples)
            eeg = torch.from_numpy(f['eeg'][idx].astype(np.float32))
            meta = torch.from_numpy(f['metadata'][idx].astype(np.float32))
            true_text = tokenizer.decode(f['input_ids'][idx].astype(np.int64), skip_special_tokens=True)

            # Inference
            pred_text, _, _ = beam_search_decoder(model, eeg, meta, edge_index, edge_attr, beam_width=5)
            
            predictions.append(pred_text)
            references.append(true_text) 
            
            if (n + 1) % 10 == 0:
                print(f"Sample {n+1}/{NUM_SAMPLES} | GT: {true_text} | Pred: {pred_text}")

        # 5. Calculate Scores
        print("\nComputing Metrics...")
        
        bleu_refs = [[r] for r in references]
        bleu_score = bleu_metric.compute(predictions=predictions, references=bleu_refs)
        rouge_score = rouge_metric.compute(predictions=predictions, references=references)

        print(f"\n=== FINAL EVALUATION ({NUM_SAMPLES} Samples) ===")
        print(f"BLEU Score: {bleu_score['bleu']:.4f}")
        print(f"ROUGE-1:    {rouge_score['rouge1']:.4f}")
        print(f"ROUGE-2:    {rouge_score['rouge2']:.4f}")
        print(f"ROUGE-L:    {rouge_score['rougeL']:.4f}")

Running on: cpu
Loading evaluation metrics...
Loading model on CPU...
Model loaded successfully.
Loading dataset...

Running inference on 1400 random samples...
Sample 10/1400 | GT: a chocalate cake with a knife | Pred: a person is cutting a watermelon
Sample 20/1400 | GT: a purple flower with green leaves on a black background | Pred: a glass of night sky
Sample 30/1400 | GT: a panda bear is sitting on a tree branch | Pred: a panda bear sitting on top of a
Sample 40/1400 | GT: a city with many tall buildings and clouds | Pred: a city with tall buildings
Sample 50/1400 | GT: a group of small rabbits in a bowl | Pred: a rabbit sitting on a a
Sample 60/1400 | GT: a pink and white cake with white frosting and icing | Pred: a glass of mushrooms drink on a
Sample 70/1400 | GT: a drummer ' s drum kit is shown in this video | Pred: a man playing a piano in front of
Sample 80/1400 | GT: a woman wearing black boxing gloves is boxing | Pred: a group of people walking down a street
Sample 90/1400

In [9]:
!pip install bert_score

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.3
    Uninstalling numpy-2.3.3:
      Successfully uninstalled numpy-2.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.44.1 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
# 1. Install the high-speed transfer library
!pip install -q hf_transfer

import os
# 2. Enable it by setting this environment variable
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("High-speed transfer enabled. Retrying download...")


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
High-speed transfer enabled. Retrying download...


In [ ]:
from transformers import AutoTokenizer, AutoModel

# Define path
save_directory = "/home/poorna/models/roberta-large"

print(f"Downloading roberta-large to {save_directory}...")
try:
    # Download Model & Tokenizer
    tokenizer = AutoTokenizer.from_pretrained("roberta-large")
    model = AutoModel.from_pretrained("roberta-large")

    # Save to local disk
    tokenizer.save_pretrained(save_directory)
    model.save_pretrained(save_directory)

    print("Success! Model saved locally.")
except Exception as e:
    print(f"Download failed: {e}")

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from torch_geometric.nn import GCNConv
from transformers import AutoTokenizer
from torch_geometric.utils import add_self_loops
import json
import os
import evaluate 
from tqdm.auto import tqdm # Added for progress bar

# ==================================================================================
# CONFIGURATION
# ==================================================================================

MODEL_WEIGHTS_PATH = "eeg-text-polished.pt" 
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

# Dimensions
NUM_COLORS = 9       
NUM_OBJECTS = 6      
ENC_HIDDEN = 256
DEC_HIDDEN = 256
DEC_LAYERS = 2
EMB_DIM = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Load Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    print("Local tokenizer not found, downloading bert-base-uncased...")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# ==================================================================================
# MODEL CLASSES
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, 
                          dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size, _, num_timesteps = eeg.shape
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        x = F.relu(self.gcn2(self.dropout(F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))), batch_edge_index, batch_edge_attr))
        
        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        return encoder_outputs.permute(1, 0, 2), encoder_hidden

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        self.color_processor = nn.Sequential(nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, color_feature_dim))
        self.object_processor = nn.Sequential(nn.Linear(num_objects, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, object_feature_dim))
        self.output_dim = color_feature_dim + object_feature_dim

    def forward(self, metadata):
        return torch.cat([self.color_processor(metadata[:, :NUM_COLORS].float()), 
                          self.object_processor(metadata[:, NUM_COLORS:].float())], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = nn.Linear(enc_hidden * 2, dec_hidden) 
        self.rnn = nn.GRU(emb_dim + enc_hidden * 2 + meta_features_dim + enc_hidden * 2, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_hidden * 2, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last = torch.cat((hidden[-1][0], hidden[-1][1]), dim=1)
        return torch.tanh(self.bridge(last)).unsqueeze(0).repeat(self.num_layers, 1, 1)

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        embedded = self.dropout(self.embedding(token.unsqueeze(0)))
        attn_energies = self.attention(encoder_outputs)
        scores = torch.bmm(decoder_hidden[-1].unsqueeze(0).permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        context = torch.bmm(F.softmax(scores, dim=2), encoder_outputs.permute(1, 0, 2)).permute(1, 0, 2)
        
        rnn_input = torch.cat((embedded, context, meta_features.unsqueeze(0), global_eeg_context.unsqueeze(0)), dim=2)
        output, hidden = self.rnn(rnn_input, decoder_hidden)
        return self.fc_out(output.squeeze(0)), hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_feature_dim=32, object_feature_dim=32, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        self.meta_encoder = MetadataEncoder(num_colors, num_objects, color_feature_dim, object_feature_dim)
        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden, self.meta_encoder.output_dim, dec_layers, pad_id, dropout)
        self.meta_head = nn.Sequential(nn.Linear(enc_hidden*2, 256), nn.ReLU(), nn.LayerNorm(256), nn.Dropout(0.3), nn.Linear(256, num_colors + num_objects))
        self.num_colors = num_colors

# ==================================================================================
# HELPERS
# ==================================================================================

def create_static_graph(num_channels=62):
    edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
    edge_attr = torch.ones(edge_index.shape[1], dtype=torch.float)
    return edge_index, edge_attr

def beam_search_decoder(model, eeg_signal, meta_signal, edge_index, edge_attr, beam_width=3, max_len=30):
    model.eval()
    with torch.no_grad():
        eeg, meta = eeg_signal.unsqueeze(0).to(device), meta_signal.unsqueeze(0).to(device)
        enc_out, enc_hid = model.encoder(eeg, edge_index, edge_attr)
        meta_feat = model.meta_encoder(meta)
        dec_hid = model.decoder.init_hidden(enc_hid)
        
        hidden_reshaped = enc_hid.view(model.encoder.rnn.num_layers, 2, 1, -1)
        global_ctx = torch.cat((hidden_reshaped[-1][0], hidden_reshaped[-1][1]), dim=1)
        
        meta_preds = model.meta_head(global_ctx)
        
        # Beam Search
        beams = [(0.0, SOS_ID, dec_hid, [])]
        
        for _ in range(max_len):
            candidates = []
            for score, inp, hid, seq in beams:
                if len(seq) > 0 and seq[-1] == EOS_ID:
                    candidates.append((score, inp, hid, seq)); continue
                
                token_tensor = torch.tensor([inp], device=device)
                pred, new_hid, _ = model.decoder(token_tensor, hid, enc_out, meta_feat, global_ctx)
                
                log_probs = F.log_softmax(pred, dim=-1).squeeze(0)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for k in range(beam_width):
                    candidates.append((score + topk_probs[k].item(), topk_ids[k].item(), new_hid, seq + [topk_ids[k].item()]))
            
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(seq[-1] == EOS_ID for _, _, _, seq in beams if len(seq) > 0): break

        best_seq = beams[0][3][:-1] if beams[0][3] and beams[0][3][-1] == EOS_ID else beams[0][3]
        return tokenizer.decode(best_seq, skip_special_tokens=True)

# ==================================================================================
# MAIN EXECUTION
# ==================================================================================
if __name__ == "__main__":
    if not os.path.exists(MODEL_WEIGHTS_PATH):
        print(f"Error: Weights file '{MODEL_WEIGHTS_PATH}' not found.")
        exit()

    # 1. Load Metrics
    print("Loading evaluation metrics...")
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")

    # 2. Load Model
    print("Loading model on CPU/GPU...")
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        enc_hidden=ENC_HIDDEN,
        dec_hidden=DEC_HIDDEN,
        emb_dim=EMB_DIM,
        dec_layers=DEC_LAYERS
    ).to(device)

    try:
        state_dict = torch.load(MODEL_WEIGHTS_PATH, map_location=device)
        model.load_state_dict(state_dict)
        print("Model loaded successfully.")
    except Exception as e:
        print(f"CRITICAL: Model mismatch error.\n{e}")
        exit()

    # 3. Graph
    edge_index, edge_attr = create_static_graph(num_channels=62)
    edge_index, edge_attr = edge_index.to(device), edge_attr.to(device)

    # 4. Inference Loop
    print("Loading dataset...")
    with h5py.File(H5_FILE_PATH, 'r') as f:
        total_samples = f['eeg'].shape[0]
        
        predictions = []
        references = []

        print(f"\nStarting Full Evaluation on {total_samples} samples.")
        print("This may take a while...")
        
        # === FIX: Iterate through ALL samples sequentially ===
        for idx in tqdm(range(total_samples), desc="Evaluating"):
            try:
                eeg = torch.from_numpy(f['eeg'][idx].astype(np.float32))
                meta = torch.from_numpy(f['metadata'][idx].astype(np.float32))
                true_text = tokenizer.decode(f['input_ids'][idx].astype(np.int64), skip_special_tokens=True)

                # Inference (Using beam_width=3 is faster for full dataset than 5)
                pred_text = beam_search_decoder(model, eeg, meta, edge_index, edge_attr, beam_width=3)
                
                predictions.append(pred_text)
                references.append(true_text) 
                
            except Exception as e:
                print(f"Skipping index {idx} due to error: {e}")
                continue

        # 5. Calculate Scores
        print("\nComputing Metrics...")
        
        bleu_refs = [[r] for r in references]
        bleu_score = bleu_metric.compute(predictions=predictions, references=bleu_refs)
        rouge_score = rouge_metric.compute(predictions=predictions, references=references)

        print(f"\n=== FINAL EVALUATION (ALL {total_samples} Samples) ===")
        print(f"BLEU Score: {bleu_score['bleu']:.4f}")
        print(f"ROUGE-1:    {rouge_score['rouge1']:.4f}")
        print(f"ROUGE-2:    {rouge_score['rouge2']:.4f}")
        print(f"ROUGE-L:    {rouge_score['rougeL']:.4f}")

Running on: cuda
Loading evaluation metrics...
Loading model on CPU/GPU...
Model loaded successfully.
Loading dataset...

Starting Full Evaluation on 28000 samples.
This may take a while...


Evaluating:   0%|          | 0/28000 [00:00<?, ?it/s]


Computing Metrics...

=== FINAL EVALUATION (ALL 28000 Samples) ===
BLEU Score: 0.1861
ROUGE-1:    0.4596
ROUGE-2:    0.2400
ROUGE-L:    0.4489
